In [ ]:
import json
import random
import re
from pathlib import Path
from typing import Literal

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from bert_score import score as bert_score

random.seed(42)
EVAL_N = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
DATA_DIR   = Path("../data/vistext_train_test")
TRAIN_PATH = DATA_DIR / "data_train.json"
TEST_PATH  = DATA_DIR / "data_test.json"

with open(TRAIN_PATH) as f:
    train_data = json.load(f)

with open(TEST_PATH) as f:
    test_data = json.load(f)

eval_data = random.sample(test_data, min(EVAL_N, len(test_data)))

print(f"Train: {len(train_data)}  |  Test: {len(test_data)}  |  Eval subset: {len(eval_data)}")
print("\nSample keys:", list(eval_data[0].keys()))

In [ ]:
def load_pipeline(model_name: str):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto",
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=False,          # greedy for reproducibility in baseline
    )
    return pipe

def generate(pipe, user_message: str) -> str:
    tokenizer = pipe.tokenizer
    if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template:
        messages = [{"role": "user", "content": user_message}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = user_message

    out = pipe(prompt, return_full_text=False)
    return out[0]["generated_text"].strip()



In [ ]:
def create_prompt(sg, type):

    if type=="minimal":
        return f""""Generate alt-text for this data visualization:\n\n{sg}"""

    elif type=="wcag":
            return (
        "You are an accessibility expert.\n"
        "Generate WCAG 2.1-compliant alt-text for the data visualization below.\n"
        "Produce two parts:\n"
        "1. SHORT: one sentence identifying chart type and subject (for the alt attribute).\n"
        "2. LONG: a paragraph encoding axis labels, scale, primary trend, and notable features "
        "(for aria-describedby).\n\n"
        f"Visualization:\n{sg}"
    )
    else:
        return ("You are an accessibility expert. Generate alt-text in two parts:\n"
        "LEVEL 1 (structural): state chart type, title, axis labels, and scale range.\n"
        "LEVEL 2-3 (analytical): describe the main trend, any extrema, correlations, "
        "and notable patterns. Do NOT add subjective editorial opinion.\n\n"
        f"Visualization:\n{sg}")